# 00 Setup: replicating this project on another computer

How to get a working copy of the project: the code, the Python packages (at the versions used here), and the
hackathon data in the folders the code expects. The two cells in step 4 check the environment and the data, so
run them once everything is in place.

1. Get the code
2. Install Python and the packages
3. Put the data in place
4. Check the setup
5. Build the database and results

## 1. Get the code

```bash
git clone https://github.com/nc1387/avangrid-hackathon.git
cd avangrid-hackathon
```

Run everything from this folder: notebooks, `python -m avangrid ...` commands and the scripts. The hackathon
data is **not** in git (`onedrive-export/Data` is in `.gitignore`, as are `*.db` files), so step 3 copies it in
separately. The four challenge PDFs in `onedrive-export/` are in git.

## 2. Install Python and the packages

The project was developed on **Python 3.13.9** (Anaconda, macOS). Create a fresh Python 3.13 environment, then
install the exact package versions from `requirements.txt` and the project's own `avangrid` package.

With conda:

```bash
conda create -n avangrid python=3.13
conda activate avangrid
pip install -r requirements.txt
pip install -e .
```

Or with venv (kept outside the project folder so it isn't picked up by git):

```bash
python3.13 -m venv ~/.venvs/avangrid
source ~/.venvs/avangrid/bin/activate
pip install -r requirements.txt
pip install -e .
```

`pip install -e .` installs `avangrid/` in editable mode, so `import avangrid` works from any folder and code
changes take effect without reinstalling.

The notebooks were saved with the kernel "Python [conda env:base]". To run them in Jupyter from your new
environment, register it as a kernel once and pick it from Jupyter's kernel menu:

```bash
python -m ipykernel install --user --name avangrid --display-name "Python (avangrid)"
```

### Library versions used

| Package | Version | Used for |
|---|---|---|
| Python | 3.13.9 | |
| duckdb | 2.0.0 | The `avangrid.db` database: raw meter data, imputation, every saved table |
| polars | 1.42.0 | Loading the meter CSVs and Excel workbooks; all table work |
| pyarrow | 21.0.0 | Moving data between DuckDB, polars and pandas |
| fastexcel | 0.20.2 | Excel reader behind `polars.read_excel` |
| numpy | 2.3.5 | The customers × days × 24 hourly arrays and all feature maths |
| scipy | 1.16.3 | Linear program that coordinates battery charging windows |
| scikit-learn | 1.7.2 | PU-learning classifiers (gradient boosting), k-means segmentation, metrics |
| joblib | 1.5.2 | Training many small models in parallel |
| threadpoolctl | 3.5.0 | Keeping each model on one thread |
| matplotlib | 3.10.6 | Charts in the notebooks and `onedrive-export/Data/eda/` |
| jupyter | 1.1.1 | Running the notebooks |
| ipykernel | 6.31.0 | The notebook kernel |
| nbformat | 5.10.4 | Reading and writing notebooks for `python -m avangrid run` |
| nbclient | 0.10.2 | Executing notebooks for `python -m avangrid run` |
| pandas | 2.3.3 | Building the Part 3 planner (`scripts/run_part3.py`) |

`requirements.txt` holds the same list; the first cell in step 4 compares it with what is installed.

## 3. Put the data in place

Copy the hackathon data from OneDrive into `onedrive-export/Data/` so the project folder looks like this:

```
avangrid-hackathon/                                  the git repository
├── 00_setup.ipynb … 99_build-planner.ipynb
├── avangrid/   scripts/   web/part3/   config/   data/   docs/   archive/
├── requirements.txt   pyproject.toml   README.md
└── onedrive-export/
    ├── Challenge Statement.pdf                      in git
    ├── Data Dictionary.pdf                          in git
    ├── Hackathon Kickoff 2026.pdf                   in git
    ├── Scoring Rubric.pdf                           in git
    └── Data/                                        copy from OneDrive (not in git)
        ├── EV_Heat Pump_Solar PV Customers.xlsx
        ├── Weather Data_Avangrid_Hackathon .xlsx    note the space before ".xlsx"
        ├── meter-data/
        │   ├── 2026 Avangrid Hackathon Data_1_10000/        part-00000-….csv … part-00051-….csv
        │   ├── 2026 Avangrid Hackathon Data_10001_20000/
        │   ├── 2026 Avangrid Hackathon Data_20001_30000/
        │   └── 2026 Avangrid Hackathon Data_30001_35725/
        │
        │   Created by the code; don't copy these:
        ├── avangrid.db      DuckDB database, about 2 GB (notebook 10 onward)
        ├── cache/           hourly arrays, about 3.5 GB; safe to delete, rebuilt on demand
        ├── eda/             charts and CSV exports
        └── results/         model comparisons and summaries (notebook 50 on) and the rebuild logs
```

Things to get right:

- **The two workbook names must match exactly**, including the space in `Weather Data_Avangrid_Hackathon .xlsx`.
  The paths are set in `avangrid/config.py`.
- **Meter data:** every `.csv` anywhere under `meter-data/` is loaded, so the subfolder names don't affect the
  results; keep the original names anyway so the folders are easy to match against OneDrive. The CSVs are
  about 13 GB.
- **Incomplete folder:** as of 13 Nov 2026, `2026 Avangrid Hackathon Data_10001_20000` has 44 of its 51 part
  files (parts 41 and 44–49 are missing, about 1,400 customers). Avangrid has been asked for the rest.
- **Data somewhere else:** to keep the data outside the project folder (e.g. on an external disk), set
  `AVANGRID_DATA_DIR` to the folder that holds the two workbooks and `meter-data/`, before starting Jupyter or
  Python:

  ```bash
  export AVANGRID_DATA_DIR="/Volumes/External/avangrid-data"
  ```

## 4. Check the setup

**Environment.** Compares the installed packages with `requirements.txt` and checks that `import avangrid`
picks up this project folder. `different` means another version is installed (results may still match, but
aren't guaranteed to); `MISSING` means `pip install -r requirements.txt` hasn't been run in this environment.

In [1]:
import platform
import re
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

print(f'Python {platform.python_version()} ({sys.executable})')
if sys.version_info[:2] != (3, 13):
    print('  -> developed on Python 3.13; other versions may not install the pinned packages')

pins = [re.match(r'([A-Za-z0-9_.-]+)==([^\s#]+)', line) for line in Path('requirements.txt').read_text().splitlines()]
pins = [(m[1], m[2]) for m in pins if m]
problems = 0
print(f"\n{'package':<16}{'required':<12}{'installed':<12}")
for name, wanted in pins:
    try:
        have = version(name)
    except PackageNotFoundError:
        have = None
    status = 'ok' if have == wanted else 'MISSING' if have is None else 'different'
    problems += status != 'ok'
    print(f"{name:<16}{wanted:<12}{have or '-':<12}{'' if status == 'ok' else status}")

import avangrid  # noqa: E402
pkg = Path(avangrid.__file__).resolve().parent.parent
print(f'\navangrid package loaded from {pkg}')
if pkg != Path.cwd().resolve():
    print('  -> not this folder: run `pip install -e .` here, or start Jupyter from this folder')
print('\nAll packages match requirements.txt' if problems == 0 else f'\n{problems} package(s) differ from requirements.txt')

Python 3.13.9 (/opt/anaconda3/bin/python)

package         required    installed   
duckdb          2.0.0       2.0.0       
polars          1.42.0      1.42.0      
pyarrow         21.0.0      21.0.0      
fastexcel       0.20.2      0.20.2      
numpy           2.3.5       2.3.5       
scipy           1.16.3      1.16.3      
scikit-learn    1.7.2       1.7.2       
joblib          1.5.2       1.5.2       
threadpoolctl   3.5.0       3.5.0       
matplotlib      3.10.6      3.10.6      
jupyter         1.1.1       1.1.1       
ipykernel       6.31.0      6.31.0      
nbformat        5.10.4      5.10.4      
nbclient        0.10.2      0.10.2      
pandas          2.3.3       2.3.3       

avangrid package loaded from /Users/nchaturvedi/Documents/Code/avangrid

All packages match requirements.txt


**Data.** Checks that the workbooks and meter CSVs are where `avangrid/config.py` expects them, counts the
part files in each meter-data folder, and creates the output folders (`cache/`, `eda/`, `results/`) if they
don't exist yet. It only reads the data and creates empty folders, so it's safe to re-run.

In [2]:
from avangrid import config

print(f'Data folder: {config.DATA_DIR}')
missing = [p for p in (config.CUSTOMER_GROUPS_PATH, config.WEATHER_PATH) if not p.exists()]
for p in (config.CUSTOMER_GROUPS_PATH, config.WEATHER_PATH):
    print(f"  {'ok     ' if p.exists() else 'MISSING'} {p.name}")

meter_dir = config.DATA_DIR / 'meter-data'
folders = sorted(d for d in meter_dir.iterdir() if d.is_dir()) if meter_dir.exists() else []
print(f'\nMeter data ({meter_dir}):')
total = 0
for d in folders:
    parts = sorted(d.glob('*.csv'))
    total += len(parts)
    gb = sum(p.stat().st_size for p in parts) / 1e9
    print(f'  {d.name:<44}{len(parts):>3} CSV files  {gb:5.1f} GB')
print(f'  {total} CSV files in total' if total else '  MISSING: no CSV files found')

for d in (config.CACHE_DIR, config.EDA_DIR, config.RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print(f"\ndatabase {'exists' if config.DB_PATH.exists() else 'not built yet (notebook 10 creates it)'}: {config.DB_PATH}")
print('\nData is in place' if not missing and total else '\nSome data is missing: see step 3')

Data folder: /Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data
  ok      EV_Heat Pump_Solar PV Customers.xlsx
  ok      Weather Data_Avangrid_Hackathon .xlsx

Meter data (/Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/meter-data):
  2026 Avangrid Hackathon Data_10001_20000     44 CSV files    3.6 GB
  2026 Avangrid Hackathon Data_1_10000         52 CSV files    4.2 GB
  2026 Avangrid Hackathon Data_20001_30000     51 CSV files    4.2 GB
  2026 Avangrid Hackathon Data_30001_35725     52 CSV files    2.4 GB
  199 CSV files in total

database exists: /Users/nchaturvedi/Documents/Code/avangrid/onedrive-export/Data/avangrid.db

Data is in place


## 5. Build the database and results

Either run the notebooks `10` to `99` in order in Jupyter, or run them all in one go from the project folder:

```bash
scripts/rebuild_all.sh
```

It runs notebooks 10–99 in order, stops at the first failure, and writes one log per notebook to
`onedrive-export/Data/results/rebuild_logs/<start time>/`. Each notebook is executed in place, so its outputs
are saved. To resume after a failure, pass the step number, e.g. `scripts/rebuild_all.sh 6` to start again
at notebook 50. **Notebook 50 holds every long-running model and takes about 2 hours** on a 15-core Apple M5
Pro with 24 GB of memory; the other notebooks take about 15 minutes together (2 h 10 min in all).

- Notebook 10 only loads the meter CSVs into `avangrid.db` when that file doesn't exist yet. To pick up new
  or changed meter data, move the old `avangrid.db` aside first.
- Close anything that has `avangrid.db` open (open notebooks or scripts) before rebuilding: DuckDB
  allows only one writer.

Once the build has finished, open the Part 3 demand-flexibility and battery planner: **`onedrive-export/Data/part3/index.html`**,
built on the real data by notebook 99 (the last step above). It opens in any browser, with no server; see
`web/part3/README.md`. It holds real customer data, so keep it inside the data folder.

See `README.md` for what each notebook, module and command does.